### Import packages

In [1]:
import pandas as pd
import numpy as np
import os
import re
from pathlib import Path
import fitz
import json

In [2]:
import re
import numpy as np
import pandas as pd

# ============================================================
# Config
# ============================================================
CHUNK_CSV = "data_files/chunked_paragraphs.csv"
GITHUB_URL = "https://raw.githubusercontent.com/US-CBO/eval-projections/main/output_data/outlay_projection_errors.csv"

OUTPUT_PARQUET = "data_files/flattened_cbo_reports_centroid_similarity.parquet"
CENTROIDS_PARQUET = "data_files/subcategory_centroids.parquet"

STRICT_THRESHOLD = 0.7
MIN_EXAMPLES_PER_SUB = 5
USE_LABEL_FALLBACK = True

PER_LABEL_THRESHOLD = {
    "Total": 0.80,
    "Total Mandatory": 0.80,
    "Total Discretionary": 0.80,
}

# ============================================================
# Load paragraphs
# ============================================================
df2 = pd.read_csv(CHUNK_CSV).reset_index(drop=True)
df2["row_id"] = np.arange(len(df2), dtype=np.int64)

for col in ["component","category","subcategory","match_method",
            "similarity_score","best_subcategory","best_similarity"]:
    if col not in df2.columns:
        df2[col] = pd.NA

if df2.empty:
    raise ValueError("df2 is empty. Check chunked_paragraphs.csv")

# ============================================================
# Load official CBO mapping (subcategory -> category/component)
# ============================================================
df_errors = pd.read_csv(GITHUB_URL)
mapping = (
    df_errors[["subcategory", "category", "component"]]
    .dropna(subset=["subcategory", "category", "component"])
    .drop_duplicates()
    .drop_duplicates(subset=["subcategory"], keep="first")
)
subcategories = sorted(mapping["subcategory"].unique().tolist())
print(f"Loaded {len(subcategories)} official subcategories.")

# ============================================================
# Guardrails / noise filters
# ============================================================
CAPTION_RE = re.compile(
    r"^\s*(figure|table|chapter)\b|^\s*notes?\b|^\s*source[s]?\b|^\s*memorandum\b",
    flags=re.IGNORECASE,
)
HOMELAND_RE = re.compile(
    r"\bhomeland security\b|\bdepartment of homeland security\b|\btransportation security\b|\baviation security\b|\btsa\b|\bterroris(m|t)\b",
    flags=re.IGNORECASE,
)
SS_ANCHOR_RE = re.compile(
    r"\bsocial security\b|\boasi\b|\boasdi\b|old-age and survivors insurance|survivors insurance|disability insurance",
    flags=re.IGNORECASE,
)

def looks_like_caption(text: str) -> bool:
    t = str(text).strip()
    if not t:
        return True
    if re.search(r"\bfigure\s+[a-z0-9\-]+|\btable\s+[a-z0-9\-]+", t, flags=re.IGNORECASE):
        return True
    if len(t) < 60 and CAPTION_RE.search(t):
        return True
    return False

# ============================================================
# Tier 1: exact match on official subcategory names
# ============================================================
def normalize_text(x):
    x = "" if pd.isna(x) else str(x).lower()
    x = re.sub(r"\s+", " ", x)
    return x.strip()

records = sorted(subcategories, key=lambda s: len(str(s)), reverse=True)
compiled_patterns = [
    (sub, re.compile(r"(?<!\w)" + re.escape(normalize_text(sub)) + r"(?!\w)", flags=re.IGNORECASE))
    for sub in records
]

def exact_subcategory_match(text: str):
    t = normalize_text(text)
    for sub, pat in compiled_patterns:
        if pat.search(t):
            return sub
    return pd.NA

print("Tier 1: exact phrase matching...")
df2["subcategory"] = df2["text"].astype(str).apply(exact_subcategory_match)
df2["match_method"] = np.where(df2["subcategory"].notna(), "literal", pd.NA)
df2["similarity_score"] = pd.NA
df2["best_subcategory"] = pd.NA
df2["best_similarity"] = pd.NA

literal_solved = int(df2["subcategory"].notna().sum())
remaining_count = int(df2["subcategory"].isna().sum())
print(f"Tier 1 matched: {literal_solved:,} | Tier 2 remaining: {remaining_count:,}")

# attach category/component (for literal matches and later centroid matches)
df2 = df2.drop(columns=["category","component"], errors="ignore").merge(mapping, on="subcategory", how="left")
assert df2["row_id"].is_unique, "row_id not unique after merge (unexpected)."

# ============================================================
# Word2Vec + embeddings (always, because we want embeddings saved)
# ============================================================
print("Loading Word2Vec model (cached after first download)...")
import gensim.downloader as api
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.notebook import tqdm

import nltk
from nltk.corpus import stopwords

try:
    stop_words = set(stopwords.words("english"))
except LookupError:
    nltk.download("stopwords", quiet=True)
    stop_words = set(stopwords.words("english"))

cbo_fillers = {"billion","million","percent","year","years","fiscal","estimate","estimated","cbo"}
stop_words.update(cbo_fillers)

w2v_model = api.load("word2vec-google-news-300")
vector_size = w2v_model.vector_size

def get_mean_vector(text, model, size):
    raw_words = [w.strip(",.?!()\"';:") for w in str(text).lower().split()]
    words = [w for w in raw_words if w and w not in stop_words]
    vecs = [model[w] for w in words if w in model]
    if not vecs:
        return np.zeros(size, dtype=np.float32)
    return np.mean(vecs, axis=0).astype(np.float32)

# embedding cache for ALL rows
all_embeddings = np.zeros((len(df2), vector_size), dtype=np.float32)

# ============================================================
# Build centroids from literal matches (using embeddings)
# ============================================================
literal_mask = (df2["match_method"] == "literal") & df2["subcategory"].notna()
df_literal = df2.loc[literal_mask, ["row_id","subcategory","text"]].copy()

print("Embedding literal matches (to build centroids)...")
literal_texts = df_literal["text"].astype(str).tolist()
literal_embeddings = np.zeros((len(df_literal), vector_size), dtype=np.float32)

chunk_size = 2000
for start in tqdm(range(0, len(literal_texts), chunk_size), desc="Embed literal"):
    chunk = literal_texts[start:start+chunk_size]
    embs = np.array([get_mean_vector(t, w2v_model, vector_size) for t in chunk], dtype=np.float32)
    literal_embeddings[start:start+len(chunk)] = embs

# store literal embeddings into full cache
literal_row_ids = df_literal["row_id"].to_numpy(dtype=np.int64)
all_embeddings[literal_row_ids] = literal_embeddings

df_literal["__row"] = np.arange(len(df_literal))
df_literal["__has_vec"] = (np.linalg.norm(literal_embeddings, axis=1) > 0)

centroids = {}
counts = {}

for sub in subcategories:
    idx = df_literal.index[df_literal["subcategory"] == sub].tolist()
    if not idx:
        counts[sub] = 0
        continue

    rows = df_literal.loc[idx, "__row"].to_numpy()
    good = df_literal.loc[idx, "__has_vec"].to_numpy()
    rows = rows[good]

    if len(rows) < MIN_EXAMPLES_PER_SUB:
        counts[sub] = int(len(rows))
        continue

    centroids[sub] = literal_embeddings[rows].mean(axis=0).astype(np.float32)
    counts[sub] = int(len(rows))

if USE_LABEL_FALLBACK:
    generic_contexts = [
        "federal outlays",
        "government spending",
        "budget outlays",
        "mandatory and discretionary spending",
        "congressional budget office outlays",
    ]
    for sub in subcategories:
        if sub in centroids:
            continue
        phrase_list = [sub] + [f"{c} {sub}" for c in generic_contexts]
        vecs = np.array([get_mean_vector(p, w2v_model, vector_size) for p in phrase_list], dtype=np.float32)
        if np.linalg.norm(vecs.mean(axis=0)) > 0:
            centroids[sub] = vecs.mean(axis=0).astype(np.float32)

usable = [s for s in subcategories if s in centroids]
if not usable:
    raise ValueError("No usable centroids were built.")

centroid_matrix = np.vstack([centroids[s] for s in usable]).astype(np.float32)

print("Centroid coverage (top 10):")
print(pd.Series(counts).sort_values(ascending=False).head(10))

# ============================================================
# Tier 2: classify remaining by similarity to centroids
# ============================================================
def allowed_assignment(label: str, text: str) -> bool:
    t = str(text)
    if looks_like_caption(t):
        return False
    if label == "Social Security":
        if HOMELAND_RE.search(t):
            return False
        return SS_ANCHOR_RE.search(t) is not None
    return True

na_mask = df2["subcategory"].isna()
na_indices = df2[na_mask].index.tolist()
na_texts = df2.loc[na_mask, "text"].astype(str).tolist()

print("Tier 2: centroid similarity...")
for start in tqdm(range(0, len(na_texts), chunk_size), desc="Tier 2"):
    chunk = na_texts[start:start+chunk_size]
    chunk_idx = na_indices[start:start+chunk_size]

    chunk_emb = np.array([get_mean_vector(t, w2v_model, vector_size) for t in chunk], dtype=np.float32)

    # store embeddings into full cache
    chunk_row_ids = df2.loc[chunk_idx, "row_id"].to_numpy(dtype=np.int64)
    all_embeddings[chunk_row_ids] = chunk_emb

    sims = cosine_similarity(chunk_emb, centroid_matrix)
    best_j = sims.argmax(axis=1)
    best_s = sims.max(axis=1)

    for row_i, text, j, score in zip(chunk_idx, chunk, best_j, best_s):
        best_sub = usable[int(j)]
        df2.at[row_i, "best_subcategory"] = best_sub
        df2.at[row_i, "best_similarity"] = float(score)

        thr = PER_LABEL_THRESHOLD.get(best_sub, STRICT_THRESHOLD)
        if score >= thr and allowed_assignment(best_sub, text):
            df2.at[row_i, "subcategory"] = best_sub
            df2.at[row_i, "match_method"] = "centroid"
            df2.at[row_i, "similarity_score"] = float(score)

# attach category/component for centroid matches too
df2 = df2.drop(columns=["category","component"], errors="ignore").merge(mapping, on="subcategory", how="left")
assert df2["row_id"].is_unique, "row_id not unique after final merge (unexpected)."

# ============================================================
# Write centroids to Parquet (NEW)
# ============================================================
df_centroids = pd.DataFrame({
    "subcategory": usable,
    "n_literal_examples_used": [int(counts.get(s, 0)) for s in usable],
    "centroid_embedding": [centroids[s].tolist() for s in usable],
})
df_centroids["embedding_model"] = "word2vec-google-news-300"
df_centroids["embedding_dim"] = int(vector_size)
df_centroids["strict_threshold"] = float(STRICT_THRESHOLD)

print(f"Writing centroids parquet: {CENTROIDS_PARQUET}")
df_centroids.to_parquet(CENTROIDS_PARQUET, index=False, engine="pyarrow", compression="zstd")

# ============================================================
# Put paragraph embeddings into df and write main Parquet
# ============================================================
df2["embedding"] = [vec.tolist() for vec in all_embeddings]
df2["embedding_model"] = "word2vec-google-news-300"
df2["embedding_dim"] = int(vector_size)

print(f"Writing main parquet: {OUTPUT_PARQUET}")
df2.to_parquet(OUTPUT_PARQUET, index=False, engine="pyarrow", compression="zstd")

print("Done.")
print(df2["match_method"].value_counts(dropna=False))
print(df2["subcategory"].value_counts(dropna=False).head(15))

Loaded 10 official subcategories.
Tier 1: exact phrase matching...
Tier 1 matched: 8,736 | Tier 2 remaining: 26,058
Loading Word2Vec model (cached after first download)...
Embedding literal matches (to build centroids)...


Embed literal:   0%|          | 0/5 [00:00<?, ?it/s]

Centroid coverage (top 10):
Total                       3454
Social Security             1987
Net Interest                 960
Medicare                     772
Medicaid                     761
Other Mandatory              259
Nondefense Discretionary     256
Total Discretionary          205
Defense Discretionary         43
Total Mandatory               39
dtype: int64
Tier 2: centroid similarity...


Tier 2:   0%|          | 0/14 [00:00<?, ?it/s]

Writing centroids parquet: data_files/subcategory_centroids.parquet
Writing main parquet: data_files/flattened_cbo_reports_centroid_similarity.parquet
Done.
match_method
<NA>        15322
centroid    10736
literal      8736
Name: count, dtype: int64
subcategory
<NA>                        15322
Total                        8373
Net Interest                 3031
Social Security              1994
Medicare                     1510
Medicaid                     1243
Nondefense Discretionary     1094
Other Mandatory               863
Total Discretionary           721
Defense Discretionary         569
Total Mandatory                74
Name: count, dtype: int64
